# LAB02 Assigment Template




### Exercise 0  Importing the census 

In [ ]:
# Import libraries necessary for this project
import numpy as np
import pandas as pd
from time import time
from IPython.display import display  # Allows the use of display() for DataFrames



# Pretty display for notebooks
%matplotlib inline

data = pd.read_csv("census.csv")

data.head()



### Exercise 1  Exploration 

In [ ]:
# The total number of records
n_records = data.shape[0]
print("Total number of records: {}".format(n_records))

In [ ]:
# The number of individuals making more than $50000 annually
n_greater_50k = (data['income'] == '>50K').sum()
print("Number of individuals making more than $50000 annually: {}".format(n_greater_50k))

In [ ]:
# The number of individuals making at most $50000 annually
n_at_most_50k = (data['income'] == '<=50K').sum()
print("Number of individuals making at most $50000 annually: {}".format(n_at_most_50k))

In [ ]:
# The percentage of individuals making at more than $50000 annually
greater_percent = (n_greater_50k / n_records) * 100
print("Percentage of individuals making more than $50000 annually: {:.2f}%".format(greater_percent))

### Exercise 2 Preprocessing 

In [ ]:
# Visualize skewed continuous features of original data
import matplotlib.pyplot as plt
import seaborn as sns

skewed = ['capital-gain', 'capital-loss']
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for index, feature in enumerate(skewed):
    sns.histplot(data[feature], bins=50, kde=False, ax=axes[index])
    axes[index].set_title(f"'{feature}' Feature Distribution")
    axes[index].set_xlabel(feature)
    axes[index].set_ylabel('Count')

plt.tight_layout()
plt.show()

In [ ]:
# Outliers Treatment
features_raw = data.drop('income', axis=1)
income_raw = data['income']

# Log-transform the skewed features to reduce outliers' impact
skewed = ['capital-gain', 'capital-loss']
features_log_transformed = features_raw.copy()
features_log_transformed[skewed] = features_log_transformed[skewed].apply(lambda x: np.log(x + 1))

# Optional clipping for extreme values (99th percentile)
for feature in skewed:
    upper_bound = features_log_transformed[feature].quantile(0.99)
    features_log_transformed[feature] = features_log_transformed[feature].clip(upper=upper_bound)

features_log_transformed[skewed].describe()

In [ ]:
# Data Transformation
from sklearn.preprocessing import MinMaxScaler

# Scale numerical features
numerical = ['age', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week']
scaler = MinMaxScaler()
features_log_minmax_transform = features_log_transformed.copy()
features_log_minmax_transform[numerical] = scaler.fit_transform(features_log_minmax_transform[numerical])

# Perform one-hot encoding on the data
features_final = pd.get_dummies(features_log_minmax_transform)
income = income_raw.apply(lambda x: 1 if x == '>50K' else 0)

print("Transformed feature columns: {}".format(features_final.shape[1]))
display(features_final.head())

### Exercise 3 Shuffle and Split Data

In [ ]:
# Split the 'feature' and 'income' data into training and testing sets
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    features_final,
    income,
    test_size=0.2,
    random_state=42,
    stratify=income
 )

# Show the results of the split
print("Training set has {} samples.".format(X_train.shape[0]))
print("Testing set has {} samples.".format(X_test.shape[0]))
print("Training positive ratio: {:.2f}%".format(y_train.mean() * 100))
print("Testing positive ratio: {:.2f}%".format(y_test.mean() * 100))

### Exercise 4 Evaluating Model
The following are some of the supervised learning models that are currently available in `scikit-learn`:
- Gaussian Naive Bayes (GaussianNB)
- Decision Trees
- Ensemble Methods (Bagging, AdaBoost, RandomForest)
- K-Nearest Neighbors
- Support Vector Machines (SVM)
- Logistic Regression
You need choose three of them, draw three ROC curves on the census data, and analyze and compare the them.

In [ ]:
# Evaluating Model
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_curve, auc, accuracy_score, f1_score
import matplotlib.pyplot as plt

models = {
    'GaussianNB': GaussianNB(),
    'RandomForest': RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    'LogisticRegression': LogisticRegression(max_iter=1000, random_state=42)
}

results = []
plt.figure(figsize=(8, 6))

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    fpr, tpr, _ = roc_curve(y_test, y_prob)
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, lw=2, label=f"{name} (AUC = {roc_auc:.3f})")

    results.append({
        'model': name,
        'accuracy': accuracy_score(y_test, y_pred),
        'f1': f1_score(y_test, y_pred),
        'auc': roc_auc
    })

plt.plot([0, 1], [0, 1], linestyle='--', color='gray', lw=1)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves for Three Models')
plt.legend(loc='lower right')
plt.grid(alpha=0.3)
plt.show()

results_df = pd.DataFrame(results).sort_values(by='auc', ascending=False).reset_index(drop=True)
display(results_df)

### Exercise 5 Evaluating Model 

In [ ]:
# Exercise 5: Analysis and comparison
from sklearn.metrics import classification_report, confusion_matrix

best_model_name = results_df.iloc[0]['model']
best_model = models[best_model_name]

print("Model comparison (sorted by AUC):")
display(results_df)

print(f"Best model by AUC: {best_model_name}")
y_best_pred = best_model.predict(X_test)

print("\nClassification Report:")
print(classification_report(y_test, y_best_pred, digits=4))

cm = confusion_matrix(y_test, y_best_pred)
cm_df = pd.DataFrame(cm, index=['Actual <=50K', 'Actual >50K'], columns=['Pred <=50K', 'Pred >50K'])
print("Confusion Matrix:")
display(cm_df)

print("\nConclusion:")
print("1) The ROC curves and AUC values show overall ranking ability.")
print("2) F1-score reflects balance between precision and recall on >50K class.")
print("3) Choose the model with higher AUC/F1 while considering simplicity and training cost.")